# RepairMate — Agentic AI Repair Diagnosis Agent
Google Colab notebook using Gemini API. Workflow: validate → safety screen → diagnose/ask a follow-up → validate output.

Create a Gemini key in [Google AI Studio](https://aistudio.google.com/app/apikey). In Colab Secrets, add `GeminiAPIKey` and enable notebook access. A hidden-input prompt is available if you do not use Secrets. Never publish your real key.

In [1]:
# CELL 1 — Fix Google Colab dependency conflict

%pip install -q google-auth==2.49.0

print("Google Colab authentication dependency fixed.")

Google Colab authentication dependency fixed.


In [1]:
import google.auth
import google.genai

print("Google Auth version:", google.auth.__version__)
print("Gemini SDK imported successfully!")

Google Auth version: 2.62.0
Gemini SDK imported successfully!


In [2]:
%pip -q install -U google-genai

## 1. Configure the API key

In [3]:
import os, json, re, getpass
from google import genai
from google.genai import types
API_KEY_ENV = "GeminiAPIKey"
MODEL_NAME = "gemini-2.5-flash"
def load_api_key():
    key = ""
    try:
        from google.colab import userdata
        key = userdata.get(API_KEY_ENV) or ""
    except Exception:
        pass
    if not key.strip():
        key = getpass.getpass(f"Enter {API_KEY_ENV} (hidden; leave blank to configure later): ").strip()
    if key: os.environ[API_KEY_ENV] = key; print("API key loaded into this runtime.")
    else: print(f"No key entered. Add a Colab Secret named {API_KEY_ENV} and rerun this cell.")
    return key
GEMINI_KEY = load_api_key()
client = genai.Client(api_key=GEMINI_KEY) if GEMINI_KEY else None
print("Client ready." if client else "Client not configured; offline safety checks still work.")

API key loaded into this runtime.
Client ready.


## 2. Define the agent

In [4]:
SYSTEM_INSTRUCTIONS = """You are RepairMate, a cautious repair-diagnosis assistant for household appliances, consumer electronics, bicycles, and common non-industrial equipment. You are not a certified technician. Return valid JSON with keys status, summary, follow_up_question, likely_causes, safe_checks, recommended_next_steps, when_to_get_help, safety_note. status is needs_info or diagnosis. Ask one concise follow-up question if key information is missing. Recommend only low-risk external checks. Never recommend opening sealed devices, working on live circuits, bypassing safety devices, handling gas lines/refrigerants, or handling swollen/damaged lithium batteries. For smoke, sparks, burning smell, gas smell, exposed wiring, electric shock, fire, or swollen battery, prioritize stopping use, moving away if needed, and contacting qualified help or emergency services. Do not invent prices, parts, warranties, or certainty. JSON only."""
DANGER_TERMS = ["gas leak", "smell gas", "gas smell", "smoke", "sparks", "burning smell", "burning odor", "exposed wire", "exposed wiring", "electric shock", "swollen battery", "battery swelling", "fire", "flames"]
def urgent_safety_report():
    return {"status":"diagnosis","summary":"This may involve an immediate safety hazard. Stop troubleshooting.","follow_up_question":"","likely_causes":[{"cause":"Potentially hazardous fault","likelihood":"Unknown","reason":"These symptoms can indicate a safety risk; the cause cannot be confirmed remotely."}],"safe_checks":["Keep away from the hazard and stop using the item if safe to do so."],"recommended_next_steps":["Do not touch exposed wires, leaking gas, hot or swollen batteries, or damaged electrical equipment.","For suspected gas, leave the area and contact local emergency/gas services from a safe location.","For smoke or fire, move away and contact local emergency services."],"when_to_get_help":["Contact an appropriate qualified professional before using the item again."],"safety_note":"Do not approach or unplug equipment if doing so would bring you near a hazard."}
def fallback_report(reason):
    return {"status":"diagnosis","summary":"The AI service is unavailable, so a diagnosis could not be generated.","follow_up_question":"","likely_causes":[{"cause":"More information needed","likelihood":"Unknown","reason":reason}],"safe_checks":["Only inspect external controls and visible indicators if safe."],"recommended_next_steps":["Retry later and record the device model and exact symptoms."],"when_to_get_help":["Contact a qualified technician if the fault persists or seems unsafe."],"safety_note":"This tool provides preliminary guidance, not a confirmed repair diagnosis."}
def parse_json_response(raw):
    raw=(raw or "").strip(); raw=re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I); raw=re.sub(r"\s*```$", "", raw)
    try: return json.loads(raw)
    except Exception:
        m=re.search(r"\{.*\}",raw,flags=re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    return None
def diagnose_repair(message, device_type="Not specified", model_number="Not provided", history=None):
    message=(message or "").strip()
    if not message: return {"error":"Please describe the device and the problem."}
    if len(message)>5000: return {"error":"Please keep the message under 5,000 characters."}
    if any(term in message.lower() for term in DANGER_TERMS): return urgent_safety_report()
    if client is None: return fallback_report(f"Set the {API_KEY_ENV} secret and rerun the setup cell.")
    history=history if isinstance(history,list) else []
    hist="\n".join(f"{x.get('role','user')}: {str(x.get('content',''))[:1500]}" for x in history[-6:] if isinstance(x,dict)) or "(none)"
    prompt=f"Device category: {device_type}\nModel number: {model_number}\nRecent conversation:\n{hist}\nLatest user message:\n{message}\nIf important information is missing, ask one follow-up question and set status to needs_info. Return JSON only."
    try:
        response=client.models.generate_content(model=MODEL_NAME,contents=prompt,config=types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTIONS,temperature=0.2,response_mime_type="application/json"))
        data=parse_json_response(response.text)
        if not isinstance(data,dict): return fallback_report("The model response was not valid JSON. Please retry.")
        defaults={"status":"diagnosis","summary":"Preliminary diagnosis only.","follow_up_question":"","likely_causes":[],"safe_checks":[],"recommended_next_steps":[],"when_to_get_help":[],"safety_note":"Stop if anything feels unsafe; consult a qualified professional."}
        for k,v in defaults.items():
            if not isinstance(data.get(k),type(v)): data[k]=v
        if data["status"] not in ("needs_info","diagnosis"): data["status"]="diagnosis"
        return data
    except Exception as exc: return fallback_report(f"Gemini request failed ({type(exc).__name__}); check API key, model access, quota, and internet.")

## 3. Test safety screening (no API key needed)

In [5]:
test_result = diagnose_repair("My device has sparks and a burning smell.")
assert test_result["status"] == "diagnosis"
assert "hazard" in test_result["summary"].lower()
print("Safety screening test passed.")
print(json.dumps(test_result, indent=2))

Safety screening test passed.
{
  "status": "diagnosis",
  "summary": "This may involve an immediate safety hazard. Stop troubleshooting.",
  "follow_up_question": "",
  "likely_causes": [
    {
      "cause": "Potentially hazardous fault",
      "likelihood": "Unknown",
      "reason": "These symptoms can indicate a safety risk; the cause cannot be confirmed remotely."
    }
  ],
  "safe_checks": [
    "Keep away from the hazard and stop using the item if safe to do so."
  ],
  "recommended_next_steps": [
    "Do not touch exposed wires, leaking gas, hot or swollen batteries, or damaged electrical equipment.",
    "For suspected gas, leave the area and contact local emergency/gas services from a safe location.",
    "For smoke or fire, move away and contact local emergency services."
  ],
  "when_to_get_help": [
    "Contact an appropriate qualified professional before using the item again."
  ],
  "safety_note": "Do not approach or unplug equipment if doing so would bring you near a 

## 4. Run an example diagnosis

In [6]:
result = diagnose_repair("My laptop turns on, but the screen stays black. The charging light is on.", device_type="Phone / computer / electronics")
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "status": "diagnosis",
  "summary": "The AI service is unavailable, so a diagnosis could not be generated.",
  "follow_up_question": "",
  "likely_causes": [
    {
      "cause": "More information needed",
      "likelihood": "Unknown",
      "reason": "Gemini request failed (ClientError); check API key, model access, quota, and internet."
    }
  ],
  "safe_checks": [
    "Only inspect external controls and visible indicators if safe."
  ],
  "recommended_next_steps": [
    "Retry later and record the device model and exact symptoms."
  ],
  "when_to_get_help": [
    "Contact a qualified technician if the fault persists or seems unsafe."
  ],
  "safety_note": "This tool provides preliminary guidance, not a confirmed repair diagnosis."
}


## 5. Interactive agent

In [7]:
def interactive_repair_agent():
    history=[]
    print("RepairMate ready. Type exit to stop.")
    while True:
        try: message=input("\nYou: ").strip()
        except (EOFError,KeyboardInterrupt): print("\nSession ended."); break
        if message.lower() in {"exit","quit"}: print("Session ended."); break
        answer=diagnose_repair(message,history=history)
        print("\nRepairMate:\n"+json.dumps(answer,indent=2,ensure_ascii=False))
        if "error" not in answer: history.extend([{"role":"user","content":message},{"role":"assistant","content":json.dumps(answer,ensure_ascii=False)}])
# Uncomment the next line to start chatting:
# interactive_repair_agent()
print("Interactive function ready. Uncomment interactive_repair_agent() when ready.")

Interactive function ready. Uncomment interactive_repair_agent() when ready.


## 6. Deploy the website on Render
The companion website is included in the project ZIP. Upload project files to GitHub without your key, create a Render Web Service/Blueprint, use build command `pip install -r requirements.txt`, start command `gunicorn app:app --bind 0.0.0.0:$PORT`, and set environment variable **GeminiAPIKey** to your key. Check `/health` after deployment.

Official guides: [Gemini setup](https://ai.google.dev/gemini-api/docs/get-started) · [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing) · [Render deploy](https://render.com/docs/your-first-deploy). Free quotas/model availability can change; Render free services may sleep when idle.